## Market basket analysis on a real world dataset

In [2]:
pip install pandas numpy matplotlib seaborn mlxtend

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 817.6 kB/s eta 0:00:02
   ----------------------- ---------------- 0.8/1.4 MB 815.5 kB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.4 MB 815.5 kB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.4 MB 815.5 kB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 688.3 kB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 688.3 kB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 688.3 kB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 688.3 kB/s eta 0:00:01
   -------------------------------------- - 1.3/1.4 MB 497.7 kB/s eta 0:00:01
   -------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
from itertools import combinations
from collections import Counter

# MINIMUM SUPPORT
MIN_SUPPORT = 0.02

# DATASET LOADING
df = pd.read_csv("OnlineRetail.csv", encoding="latin1")

# CLEAN DATA
df = df.dropna(subset=["Description"])
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[df["Country"] == "United Kingdom"]

# CREATING BASKETS
baskets = df.groupby("InvoiceNo")["Description"].apply(set)

# COUNT PRODUCTS
n = len(baskets)
item_count = Counter()

for basket in baskets:
    for item in basket:
        item_count[item] += 1

# FIND FREQUENT PRODUCTS
frequent = set()

for item, count in item_count.items():
    if count / n >= MIN_SUPPORT:
        frequent.add(item)

# COUNT PRODUCT PAIRS
pair_count = Counter()

for basket in baskets:
    items = sorted(basket & frequent)
    for pair in combinations(items, 2):
        pair_count[pair] += 1

# GENERATE ASSOCIATION RULES
rules = []

for (a, b), count in pair_count.items():

    support = count / n

    if support >= MIN_SUPPORT:

        # A -> B
        confidence = count / item_count[a]
        lift = confidence / (item_count[b] / n)

        rules.append([a, b, support, confidence, lift])

        # B -> A
        confidence = count / item_count[b]
        lift = confidence / (item_count[a] / n)

        rules.append([b, a, support, confidence, lift])

# DISPLAY RESULTS
rules = pd.DataFrame(
    rules,
    columns=["If Bought", "Then Bought", "Support", "Confidence", "Lift"]
)

rules = rules.sort_values("Lift", ascending=False)

print("\n" + "=" * 106)
print("                                    TOP 10 MARKET BASKET ASSOCIATION RULES")
print("=" * 106)

print(f"{'If Bought':<35} {'Then Bought':<35} {'Support':>10} {'Confidence':>12} {'Lift':>10}")
print("-" * 106)

for _, row in rules.head(10).iterrows():
    print(f"{row['If Bought'][:34]:<35} "
          f"{row['Then Bought'][:34]:<35} "
          f"{row['Support']:>10.3f} "
          f"{row['Confidence']:>12.3f} "
          f"{row['Lift']:>10.3f}")

print("=" * 106)


                                    TOP 10 MARKET BASKET ASSOCIATION RULES
If Bought                           Then Bought                            Support   Confidence       Lift
----------------------------------------------------------------------------------------------------------
WOODEN HEART CHRISTMAS SCANDINAVIA  WOODEN STAR CHRISTMAS SCANDINAVIAN       0.020        0.723     27.197
WOODEN STAR CHRISTMAS SCANDINAVIAN  WOODEN HEART CHRISTMAS SCANDINAVIA       0.020        0.768     27.197
GREEN REGENCY TEACUP AND SAUCER     PINK REGENCY TEACUP AND SAUCER           0.032        0.618     15.864
PINK REGENCY TEACUP AND SAUCER      GREEN REGENCY TEACUP AND SAUCER          0.032        0.821     15.864
SPACEBOY LUNCH BOX                  DOLLY GIRL LUNCH BOX                     0.024        0.608     15.666
DOLLY GIRL LUNCH BOX                SPACEBOY LUNCH BOX                       0.024        0.609     15.666
WOODLAND CHARLOTTE BAG              STRAWBERRY CHARLOTTE BAG        